# Open Storage Network (OSN): browse, stage, analyze, and upload

This guided tutorial can be run on a laptop or a computing cluster like Unity. Run the code cells in order. Most exercises read publicly available data and require no OSN credentials, while the final upload requires credentials with write permissions. We'll show you where to find those credentials if you belong to an active OSN project. 

**Data:** We'll analyze monthly ERA5 geopotential for March 2024 at  500 hPa over 65–75°N and 0–30°E. The source file contains data for 1990–2025 and is approximately **1.52 GB**, so allow sufficient disk space and transfer time. 

## 1. What is OSN?

The Open Storage Network gives researchers a place to store and share large datasets without keeping everything on a laptop or computing cluster. You can browse available files through Globus, download them for analysis, or use Python to read data directly from OSN. You can also upload data and analysis results to a project where you have write permission. 

Project files are stored in a **bucket**, which you can think of as shared storage spaces. To access a file with Python, you'll provide the OSN server address (the **endpoint**), the bucket name, and the path to the file. 

## 2. Sign in and find your project

**URL**: https://coldfront.osn.mghpcc.org/

1. **Sign in through your institution.**

2. **Find a project you belong to and select its bucket allocation.**

<img src="Screenshots/Page-Overview.png"
     alt=""
     width="800">

3. **Locate the endpoint, bucket name, and access/secret keys.** These keys (i.e., the read/write or RW access keys) authenticate your access to the bucket. You'll use them later in the upload exercise, so locate them now and keep them private. 

<img src="Screenshots/Allocation-Page.png"
     alt=""
     width="800">

## 3. Browse with Globus

**Globus provides a convenient way to browse and transfer research data between storage systems, computing clusters, and personal computers.** 

1. **Sign in to Globus using your institutional login (URL: https://www.globus.org).**

2. **Open the File Manager and search for 'UMass Amherst OSN Pod' in the Collection field.** You may be prompted to authenticate before accessing the collection. 

<img src="Screenshots/Globus-Page.png"
     alt=""
     width="800">

3. **Open the collection and navigate to the monthly geopotential file used in this tutorial.** Find the folder matching the OSN bucket name ('casadataportal-01') and follow this path:

'ERA5/Example/ERA5_pressure_geopotential_11lev_0p50deg_19900101_20251231.nc'

<img src="Screenshots/Example-Path.png"
     alt=""
     width="800">

## 4. Python setup

To run this notebook yourself, create a dedicated conda environment containing the required Python libraries. This keeps the tutorial’s software separate from your existing analysis environments.

Open a terminal, navigate to the folder containing `environment.yml` and `OSN_Tutorial.ipynb`, and run:

```bash
conda env create -f environment.yml
conda activate osn-tutorial
python -m ipykernel install --user --name osn-tutorial --display-name "Python (OSN Tutorial)"
jupyter lab OSN_Tutorial.ipynb
```

These commands create the environment, activate it, register it as a Jupyter kernel, and open the notebook. Select **Python (OSN Tutorial)** from the notebook’s kernel menu. Creating the environment and registering the kernel are one-time setup steps.

On Unity, use your usual Jupyter launch process and select the tutorial kernel instead of running the final `jupyter lab` command.

The environment includes three libraries central to this tutorial:

| Library | Purpose |
|---|---|
| **s3fs** | Connect to OSN to read, download, and upload files |
| **xarray** | Select and analyze scientific data stored in NetCDF files |
| **h5netcdf** | Enable xarray to read NetCDF4 files locally or through a remote file handle |

Run the next code cell to check that the required libraries are available. If you’re following the demonstration without running the code, no setup is needed.


In [1]:
import sys
from pathlib import Path
from time import perf_counter
from getpass import getpass

import s3fs
import xarray as xr
import h5netcdf

print("Python:", sys.executable)
print("Working directory:", Path.cwd())
print("Required libraries imported successfully.")


Python: /Users/kevinboyd/enter/envs/osn-tutorial/bin/python
Working directory: /Users/kevinboyd/UMass/OSN-Example/osn_tutorial
Required libraries imported successfully.


### Configuration and public connection

The cell below specifies the OSN endpoint and source file, sets the destination for a staged copy (`./Data`), and configures anonymous access to this public dataset.

It also defines two functions used in both the staged and remote examples:

- **`analyze(ds)`** selects March 2024 geopotential at 500 hPa over 65–75°N, 0–30°E and converts it to geopotential height.
- **`report(height)`** prints the selected grid dimensions and minimum and maximum heights.

Using the same functions ensures that we compare access methods while keeping the calculation identical. This cell only defines the configuration and functions; subsequent cells read the data and run the analysis.


In [2]:
# 1. Specify the OSN server and source file.
ENDPOINT = "https://uma1.osn.mghpcc.org"
OBJECT = (
    "casadataportal-01/ERA5/Atmosphere/Monthly/"
    "ERA5_pressure_geopotential_11lev_0p50deg_19900101_20251231.nc"
)

# 2. Set the destination for the downloaded copy.
DATA_DIR = Path("Data")
LOCAL_FILE = DATA_DIR / Path(OBJECT).name

# 3. Configure anonymous access to this public OSN file.
public_fs = s3fs.S3FileSystem(
    anon=True,
    client_kwargs={"endpoint_url": ENDPOINT},
)

# 4. Define the same analysis for both local and remote access.
def analyze(ds):
    # Select March 2024, 500 hPa, and the region of interest.
    z = ds["z"].sel(
        valid_time="2024-03-01",  # Monthly mean
        pressure_level=500,      # hPa
        latitude=slice(75, 65),   # Latitude coordinates run north to south
        longitude=slice(0, 30),
    ).load()

    # Convert geopotential to geopotential height in metres.
    height = (z / 9.80665).rename("geopotential_height")
    height.attrs = {"units": "m", "long_name": "Geopotential height"}
    return height

# 5. Define how to summarize the analysis results.
def report(height):
    print("Grid:", dict(height.sizes))
    print(f"Minimum: {height.min().item():.1f} m")
    print(f"Maximum: {height.max().item():.1f} m")


## 5. Stage the file, then analyze locally

This cell downloads the complete source file into `./Data` using **`public_fs.get()`**, then opens the local copy and runs our geopotential-height calculation.

We time the download and local analysis separately. Running this cell again downloads the file again; the next cell shows how to reuse the existing copy.


In [3]:
# 1. Download the complete OSN file into ./Data.
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Downloading to {LOCAL_FILE}", flush=True)
start = perf_counter()

public_fs.get(OBJECT, str(LOCAL_FILE))

download_seconds = perf_counter() - start

# 2. Open the local copy and run the analysis.
start = perf_counter()

with xr.open_dataset(LOCAL_FILE, engine="h5netcdf") as ds:
    staged_height = analyze(ds)

local_seconds = perf_counter() - start

# 3. Display the results and elapsed times.
staged_total_seconds = download_seconds + local_seconds

report(staged_height)
print(f"Download: {download_seconds:.2f} s")
print(f"Local read + analysis: {local_seconds:.2f} s")
print(f"Total staged workflow: {staged_total_seconds:.2f} s")

Grid: {'latitude': 21, 'longitude': 61}
Minimum: 5197.9 m
Maximum: 5403.8 m
Download: 15.70 s
Local read + analysis: 0.08 s
Total staged workflow: 15.77 s


### Analyze the staged file again

Once the file is downloaded, we can reuse it for additional analyses without downloading it again. The next cell repeats the calculation using the existing copy in ./Data and measures the elapsed time.


In [4]:
# 1. Open the existing local file and repeat the analysis.
start = perf_counter()

with xr.open_dataset(LOCAL_FILE, engine="h5netcdf") as ds:
    reused_height = analyze(ds)

reuse_seconds = perf_counter() - start

# 2. Display the results and elapsed time.
report(reused_height)
print(f"Reusing the local copy: {reuse_seconds:.2f} s")


Grid: {'latitude': 21, 'longitude': 61}
Minimum: 5197.9 m
Maximum: 5403.8 m
Reusing the local copy: 0.03 s


## 6. Read directly from OSN

We now repeat the same analysis by opening the file directly on OSN with **`public_fs.open()`**, rather than reading the downloaded copy.

The analysis selects the region, month, and pressure level we need, then loads those data into memory. Data still travel over the network, but we do not save a complete local copy of the source file.

We’ll time this approach and compare it with downloading the file first.


In [5]:
# 1. Open the file on OSN and run the analysis.
start = perf_counter()

with public_fs.open(OBJECT, "rb") as remote:
    with xr.open_dataset(remote, engine="h5netcdf") as ds:
        remote_height = analyze(ds)

remote_seconds = perf_counter() - start

# 2. Display the results and elapsed time.
report(remote_height)
print(f"Remote read + analysis: {remote_seconds:.2f} s")


Grid: {'latitude': 21, 'longitude': 61}
Minimum: 5197.9 m
Maximum: 5403.8 m
Remote read + analysis: 19.67 s


## 7. Compare results and elapsed time

In [6]:
# 1. Check that staged and remote access produced matching results.
# This raises an error if the values differ.
xr.testing.assert_allclose(staged_height, remote_height)
print("The staged and remote results agree.\n")

# 2. Collect the timings measured in the previous cells.
timings = [
    ("Download", download_seconds),
    ("Local read + analysis", local_seconds),
    ("Staged total", staged_total_seconds),
    ("Reuse staged file", reuse_seconds),
    ("Remote read + analysis", remote_seconds),
]

# 3. Print the timings side by side in an aligned list.
for label, seconds in timings:
    print(f"{label:28s} {seconds:8.2f} seconds")

The staged and remote results agree.

Download                        15.70 seconds
Local read + analysis            0.08 seconds
Staged total                    15.77 seconds
Reuse staged file                0.03 seconds
Remote read + analysis          19.67 seconds


### Comparing the time required

Here are my results (yours will likely vary): 

Both approaches produced the same result, but **staging was faster in this run**: downloading the file and analyzing it locally took **15.77 seconds**, compared with **19.67 seconds** for remote access.

Almost all of the staged workflow’s time was spent downloading the file (**15.70 seconds**). The local analysis took only **0.08 seconds**, and repeating it using the downloaded copy took **0.03 seconds**.

The distinction is therefore between **accessing data without a complete local copy** and **reusing data efficiently**:

- **Remote access** avoided the full-file download, although it took longer to complete this analysis.
- **Staging** required downloading the file first, but was faster overall in this run and made subsequent analysis of the local copy very fast.

For a one-off analysis of a small subset, remote access can be a good choice, especially when local storage is limited. For repeated work on the same file, staging is often preferable.

## 8. Save the analysis result

We’ll save the March 2024 regional geopotential-height field calculated above as a NetCDF file in `./Data`. In the next section, we’ll upload this result to OSN.

This illustrates a common workflow: read source data from OSN, perform an analysis, and upload the resulting dataset for storage or sharing.


In [7]:
# 1. Set the output filename in ./Data.
DATA_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_FILE = DATA_DIR / "ERA5_March2024_500hPa_regional_height.nc"

# 2. Save the calculated geopotential-height field as a NetCDF file.
# This contains the selected region, month, and pressure level.
remote_height.to_dataset().to_netcdf(UPLOAD_FILE, engine="h5netcdf")

# 3. Confirm where the file was saved and show its size.
print("Saved:", UPLOAD_FILE.resolve())
print(f"File size: {UPLOAD_FILE.stat().st_size:,} bytes")

Saved: /Users/kevinboyd/UMass/OSN-Example/osn_tutorial/Data/ERA5_March2024_500hPa_regional_height.nc
File size: 19,977 bytes


## 9. Authenticate and upload (optional)

To upload our result, we need OSN credentials with write permission. Return to the OSN portal shown earlier and locate your **RW access key and secret key.**

PCAR members with write access to casadataportal-01 can use the shared ERA5/Example directory. If you don’t have write access to this bucket, use a destination you’re authorized to write to or follow the demonstration without running the upload cells.

The next cell uses getpass to prompt for your keys without displaying them. Enter each key and press Enter. Keep the keys out of your code and printed outputs, and restart the kernel when you’re finished to clear the credentials held in memory.


In [8]:
# 1. Set the shared upload destination.
UPLOAD_PREFIX = "casadataportal-01/ERA5/Example"

# 2. Enter your OSN credentials using hidden prompts.
authenticated_fs = s3fs.S3FileSystem(
    anon=False,
    key=getpass("OSN access key: "),
    secret=getpass("OSN secret key: "),
    client_kwargs={"endpoint_url": ENDPOINT},
)

# 3. Display the destination.
print("Upload destination:", UPLOAD_PREFIX)
print("Write permission will be checked during upload.")


OSN access key:  ········
OSN secret key:  ········


Upload destination: casadataportal-01/ERA5/Example
Write permission will be checked during upload.


### Upload the result to OSN

The next cell uploads our saved NetCDF file using **`authenticated_fs.put()`** and reports the elapsed time.

The file is saved under `casadataportal-01/ERA5/Example`. If a file with the same name already exists, it will be replaced. Your local copy remains in `./Data`.

Uploading with credentials does not make the file private; its visibility depends on the destination’s access settings.


In [9]:
# 1. Set the full destination path on OSN.
destination = f"{UPLOAD_PREFIX.rstrip('/')}/{UPLOAD_FILE.name}"

# 2. Upload the local file, replacing any existing file with the same name.
start = perf_counter()

authenticated_fs.put(str(UPLOAD_FILE), destination)

upload_seconds = perf_counter() - start

# 3. Display the destination and elapsed time.
print("Uploaded to:", destination)
print(f"Upload time: {upload_seconds:.2f} s")
print("The local copy remains in:", UPLOAD_FILE)

Uploaded to: casadataportal-01/ERA5/Example/ERA5_March2024_500hPa_regional_height.nc
Upload time: 0.41 s
The local copy remains in: Data/ERA5_March2024_500hPa_regional_height.nc


## 10. Takeaways

If the upload completed successfully, your analysis result is now on OSN! You can return to Globus and refresh the destination folder to find it. If you received an error, possible causes include incorrect credentials, using RO instead of RW keys, a missing local file, or a connection problem.

Throughout this tutorial, we used the Python library **s3fs** to connect to OSN. The connection objects we named `public_fs` and `authenticated_fs` provide three key methods:

- **`fs.get(...)`** downloads a file from OSN to the machine running Python, staging it for local analysis.
- **`fs.open(...)`** opens a remote file so we can read data without first saving a complete local copy.
- **`fs.put(...)`** uploads a local file to OSN, replacing an existing file at the same destination.

Public reads can use anonymous access. Private reads require credentials with read permission, while uploads require **RW credentials**. 